# PPE Detection Model Training
Train YOLOv8 on Construction-PPE dataset for unsafe behavior detection.

Classes: helmet, no_helmet, vest, gloves, boots, goggles, Person, etc.

**Make sure to select GPU runtime**: Runtime -> Change runtime type -> T4 GPU

In [ ]:
# Step 1: Check GPU
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Step 2: Install Ultralytics
!pip install ultralytics==8.3.40
from ultralytics import YOLO
print('Ultralytics installed successfully')

In [ ]:
# Step 3: Train YOLOv8n on Construction-PPE dataset
# Dataset auto-downloads (~178MB), 11 classes, 1416 images
# Training ~30-40 min on T4 GPU

model = YOLO('yolov8n.pt')

results = model.train(
    data='construction-ppe.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    optimizer='auto',
    lr0=0.001,
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    project='runs/detect',
    name='ppe_train',
    verbose=True
)

print('\nTraining complete!')

In [ ]:
# Step 4: Validate on test set

best_model = YOLO('runs/detect/ppe_train/weights/best.pt')
metrics = best_model.val(data='construction-ppe.yaml', split='test')

print(f'\nmAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

In [ ]:
# Step 5: Test with a sample image from the dataset

import os
test_dir = 'datasets/construction-ppe/images/test'
test_images = [f for f in os.listdir(test_dir) if f.endswith(('.jpg', '.png'))][:3]

for img_name in test_images:
    img_path = os.path.join(test_dir, img_name)
    result = best_model.predict(img_path, conf=0.5, save=True)
    print(f'\n{img_name}:')
    for box in result[0].boxes:
        cls = result[0].names[int(box.cls)]
        conf = float(box.conf)
        print(f'  {cls}: {conf:.2f}')

In [ ]:
# Step 6: Download the trained model

from google.colab import files

# Download best.pt
files.download('runs/detect/ppe_train/weights/best.pt')
print('Downloaded: best.pt')
print('\nUpload this file to your server:')
print('  scp best.pt root@SERVER_IP:/opt/machineVision/models/yolov8-ppe.pt')